# DATA ANALYSIS 
## DATA COVERAGE 

### 3. FILTERING
#### a. Method for Identifying Potential Indicator Replication and Cross-Source Overlap

**Purpose**

This workflow identifies potential replication, duplication, and conceptual overlap among indicators in a combined country-availability dataset. The method is designed for **human review**, not automatic deletion. It creates comparison files that help reviewers decide whether two indicators should both be retained, merged conceptually, or one should be excluded.

The procedure supports two review tasks:

1. **Within-source replication**: potential duplicates within the same source family, such as `imf` versus `imf`.
2. **Cross-source overlap**: potential overlap across source families, such as `imf` versus `wb`, `imf` versus `unsd`, or `unsd` versus `wb`.

This document focuses especially on recreating the file:

```text
cross_source_overlap_candidates.csv
```

**Core Logic**

The workflow compares each pair of indicators using four main dimensions:

| Dimension | Measure | Purpose |
|---|---|---|
| Indicator text | `indicator_similarity_no_timing` | Compares indicator names after timing words and years are removed |
| Indicator plus definition | `text_similarity_no_timing` | Compares broader meaning using indicator name plus definition |
| Country coverage | `coverage_jaccard` | Measures overlap in available country sets |
| Country coverage containment | `coverage_containment` | Checks whether the smaller country set is mostly contained in the larger one |

The method also reports:

| Field | Purpose |
|---|---|
| `unit_match` | Checks whether the unit values are identical |
| `shared_terms` | Shows which substantive words are shared |
| `timing_terms_a`, `timing_terms_b` | Shows whether the difference may be mainly timing-related |
| `review_note` | Reminds reviewers that the pair requires human judgment |


*Define terms to remove*

Timing words and years are removed to detect indicators that differ mainly by measurement period, such as annual, latest year, baseline, target, or projection.

timing_terms = {
    "year", "years", "annual", "annually",
    "latest", "recent", "current", "historical",
    "time", "period", "periods",
    "baseline", "base",
    "projected", "projection", "projections",
    "forecast", "forecasts",
    "target", "targets",
    "estimate", "estimates", "estimated",
    "measured", "measurement",
    "data"
}

stop_terms = {
    "the", "and", "or", "of", "in", "to", "for", "from",
    "by", "with", "as", "a", "an", "per",
    "total", "share", "percentage", "percent",
    "number", "amount", "rate", "index",
    "value", "values",
    "country", "countries",
    "indicator", "indicators"
}
 
 *Build TF-IDF matrix for text similarity*

The workflow uses term frequency-inverse document frequency (TF-IDF) on the combined indicator and definition text after timing words are removed. It uses one-word and two-word terms.
*Set cross-source screening thresholds*

These thresholds are intentionally broad. The goal is to produce a human-review file, not an automatic deletion file.

min_indicator_similarity = 0.60
min_text_similarity = 0.55
min_coverage_jaccard = 0.25
min_coverage_containment = 0.70
min_overlap_country_count = 20
min_shared_terms = 2
 
*How to Interpret the Output*

Each row in `cross_source_overlap_candidates.csv` is one possible overlap pair across two source families.

| Column | Interpretation |
|---|---|
| `source_pair` | Source comparison, such as `imf-wb` |
| `indicator_a`, `indicator_b` | Indicator names being compared |
| `definition_a`, `definition_b` | Definitions used for human review |
| `unit_a`, `unit_b` | Units to check whether indicators measure the same quantity |
| `indicator_similarity_no_timing` | Similarity of indicator names after timing terms are removed |
| `text_similarity_no_timing` | TF-IDF cosine similarity of indicator plus definition |
| `coverage_jaccard` | Country overlap divided by country union |
| `coverage_containment` | Country overlap divided by the smaller country set |
| `overlap_country_count` | Number of countries available in both indicators |
| `unit_match` | Whether units match exactly |
| `shared_terms` | Substantive words shared by both indicators |
| `timing_terms_a`, `timing_terms_b` | Timing-related terms found in each indicator |
| `review_note` | Reminder that the pair requires human judgment |

*Important Limitations*

This method does not prove that two indicators are duplicates. It only creates a prioritized screening file.

Final decisions should be based on:

1. Indicator name.
2. Definition.
3. Unit.
4. Source description.
5. Country coverage.
6. Timing of measurement.
7. Whether the indicator measures a stock, flow, action, or policy guide.
8. Whether one indicator is more continuous, more recent, or better documented than the other.



In [38]:
# ============================================================
# Recreate cross-source overlap candidates
# Input:
#   combined_country_availability_wide.csv
#
# Output:
#   cross_source_overlap_candidates_recreated.csv
# ============================================================

import os
import re
import unicodedata
import numpy as np
import pandas as pd
from itertools import combinations
from difflib import SequenceMatcher
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

In [39]:
# ============================================================
# 1. Paths
# ============================================================

input_file = "combined_country_availability_wide.csv"

output_file = "cross_source_overlap_candidates_recreated.csv"

print("Input file:", input_file)
print("Output file:", output_file)

Input file: combined_country_availability_wide.csv
Output file: cross_source_overlap_candidates_recreated.csv


In [43]:
# ============================================================
# 2. Load data
# ============================================================

df = pd.read_csv(input_file, low_memory=False)

print("Input shape:")
print(df.shape)

print("Columns:")
print(df.columns[:15].tolist())

Input shape:
(1323, 225)
Columns:
['no', 'indicator', 'Definition', 'unit', 'source', 'original', 'dataset', 'Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Andorra', 'Angola', 'Antigua and Barbuda', 'Argentina']


In [44]:
# ============================================================
# 3. Identify metadata and country columns
# ============================================================

metadata_columns = [
    "no",
    "indicator",
    "Definition",
    "unit",
    "source",
    "original",
    "dataset"
]

country_columns = [
    col
    for col in df.columns
    if col not in metadata_columns
]

print("Number of country columns:")
print(len(country_columns))

print("First few country columns:")
print(country_columns[:10])

Number of country columns:
218
First few country columns:
['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Andorra', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba']


In [45]:
# ============================================================
# 4. Text cleaning helpers
# ============================================================

def clean_text(value):
    """
    Convert text to a clean lowercase string.
    """
    if pd.isna(value):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value).strip()
    )


def normalize_ascii(value):
    """
    Normalize accented characters and remove non-ASCII artifacts.
    """
    value = clean_text(value)

    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")

    return value


timing_terms = {
    "year", "years", "annual", "annually",
    "latest", "recent", "current", "historical",
    "time", "period", "periods",
    "baseline", "base",
    "projected", "projection", "projections",
    "forecast", "forecasts",
    "target", "targets",
    "estimate", "estimates", "estimated",
    "measured", "measurement",
    "data"
}

# Add year-like terms from 1900 to 2100
for year in range(1900, 2101):
    timing_terms.add(str(year))


stop_terms = {
    "the", "and", "or", "of", "in", "to", "for", "from",
    "by", "with", "as", "a", "an", "per",
    "total", "share", "percentage", "percent",
    "number", "amount", "rate", "index",
    "value", "values",
    "country", "countries",
    "indicator", "indicators"
}


def tokenize_text(value, remove_timing=True):
    """
    Convert text to normalized tokens.
    """
    text = normalize_ascii(value).lower()

    text = re.sub(r"[^a-z0-9]+", " ", text)

    tokens = []

    for token in text.split():
        if len(token) <= 1:
            continue

        if remove_timing:
            if token in timing_terms:
                continue

            if re.fullmatch(r"\d{4}", token):
                continue

        if token in stop_terms:
            continue

        tokens.append(token)

    return tokens


def normalized_text(value, remove_timing=True):
    """
    Return normalized text after token cleaning.
    """
    return " ".join(tokenize_text(value, remove_timing=remove_timing))


def token_overlap_similarity(text_a, text_b):
    """
    Similarity based on token overlap.

    Formula:
      overlap / min(number of unique tokens in A, number of unique tokens in B)

    This is useful when one indicator is a shorter version of another.
    Example:
      'land area covered by agricultural area'
      versus
      'land area'
    """
    tokens_a = set(tokenize_text(text_a, remove_timing=True))
    tokens_b = set(tokenize_text(text_b, remove_timing=True))

    if len(tokens_a) == 0 or len(tokens_b) == 0:
        return 0.0

    overlap = len(tokens_a.intersection(tokens_b))
    denominator = min(len(tokens_a), len(tokens_b))

    return overlap / denominator


def shared_terms(text_a, text_b):
    """
    Return shared cleaned terms between two text fields.
    """
    tokens_a = set(tokenize_text(text_a, remove_timing=True))
    tokens_b = set(tokenize_text(text_b, remove_timing=True))

    shared = sorted(tokens_a.intersection(tokens_b))

    return "; ".join(shared)


def extract_timing_terms(value):
    """
    Extract timing-related terms from text.
    """
    tokens = tokenize_text(value, remove_timing=False)

    found = []

    for token in tokens:
        if token in timing_terms or re.fullmatch(r"\d{4}", token):
            if token not in found:
                found.append(token)

    return "; ".join(found)

In [46]:
# ============================================================
# 5. Country coverage helpers
# ============================================================

def available_country_set(row, country_columns):
    """
    Return the set of countries coded as available, meaning value == 1.
    """
    available = set()

    for country in country_columns:
        value = row[country]

        try:
            numeric_value = float(value)
        except Exception:
            numeric_value = 0

        if numeric_value == 1:
            available.add(country)

    return available


def coverage_scores(set_a, set_b):
    """
    Compute country-coverage overlap measures.

    coverage_jaccard:
      intersection / union

    coverage_containment:
      intersection / smaller set

    containment is useful when one source covers fewer countries
    but those countries are almost all included in the other source.
    """
    if len(set_a) == 0 and len(set_b) == 0:
        return 0.0, 0.0, 0

    intersection_count = len(set_a.intersection(set_b))
    union_count = len(set_a.union(set_b))

    if union_count == 0:
        jaccard = 0.0
    else:
        jaccard = intersection_count / union_count

    smaller_count = min(len(set_a), len(set_b))

    if smaller_count == 0:
        containment = 0.0
    else:
        containment = intersection_count / smaller_count

    return jaccard, containment, intersection_count

In [47]:
# ============================================================
# 6. Prepare comparison fields
# ============================================================

df = df.copy()

df["indicator_clean"] = df["indicator"].map(clean_text)
df["definition_clean"] = df["Definition"].map(clean_text)
df["unit_clean"] = df["unit"].map(clean_text)
df["source_clean"] = df["source"].map(clean_text)

df["indicator_no_timing"] = df["indicator_clean"].map(
    lambda x: normalized_text(x, remove_timing=True)
)

df["full_text_no_timing"] = (
    df["indicator_clean"].fillna("")
    + " "
    + df["definition_clean"].fillna("")
).map(
    lambda x: normalized_text(x, remove_timing=True)
)

df["timing_terms"] = (
    df["indicator_clean"].fillna("")
    + " "
    + df["definition_clean"].fillna("")
).map(extract_timing_terms)

df["available_countries"] = df.apply(
    lambda row: available_country_set(row, country_columns),
    axis=1
)

df["country_count"] = df["available_countries"].map(len)

print("Prepared dataframe shape:")
print(df.shape)

print("Dataset counts:")
print(df["dataset"].value_counts())

Prepared dataframe shape:
(1323, 234)
Dataset counts:
dataset
imf     1089
unsd     158
wb        76
Name: count, dtype: int64


In [48]:
# ============================================================
# 7. Build TF-IDF similarity matrix for full text
# ============================================================

# TF-IDF is used for indicator + definition similarity.
# The token overlap measure is used separately for indicator-name similarity.

vectorizer = TfidfVectorizer(
    min_df=1,
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(
    df["full_text_no_timing"].fillna("")
)

print("TF-IDF matrix shape:")
print(tfidf_matrix.shape)

TF-IDF matrix shape:
(1323, 7311)


In [49]:
# ============================================================
# 8. Cross-source comparison settings
# ============================================================

# These thresholds are intentionally broad.
# The goal is to generate a human-review file, not automatic deletion.

min_indicator_similarity = 0.60
min_text_similarity = 0.55
min_coverage_jaccard = 0.25
min_coverage_containment = 0.70
min_overlap_country_count = 20
min_shared_terms = 2

print("Cross-source screening thresholds:")
print("min_indicator_similarity:", min_indicator_similarity)
print("min_text_similarity:", min_text_similarity)
print("min_coverage_jaccard:", min_coverage_jaccard)
print("min_coverage_containment:", min_coverage_containment)
print("min_overlap_country_count:", min_overlap_country_count)
print("min_shared_terms:", min_shared_terms)

Cross-source screening thresholds:
min_indicator_similarity: 0.6
min_text_similarity: 0.55
min_coverage_jaccard: 0.25
min_coverage_containment: 0.7
min_overlap_country_count: 20
min_shared_terms: 2


In [50]:
# ============================================================
# 9. Create cross-source overlap candidates
# ============================================================

cross_source_rows = []
pair_id = 0

datasets = sorted(df["dataset"].dropna().unique().tolist())

print("Datasets found:")
print(datasets)

for dataset_a, dataset_b in combinations(datasets, 2):

    df_a = df[df["dataset"] == dataset_a].copy()
    df_b = df[df["dataset"] == dataset_b].copy()

    print("Comparing:", dataset_a, "vs", dataset_b)
    print("Rows:", len(df_a), "x", len(df_b))

    for idx_a, row_a in tqdm(
        df_a.iterrows(),
        total=len(df_a),
        desc=f"{dataset_a} vs {dataset_b}"
    ):
        for idx_b, row_b in df_b.iterrows():

            indicator_similarity = token_overlap_similarity(
                row_a["indicator_clean"],
                row_b["indicator_clean"]
            )

            text_similarity = cosine_similarity(
                tfidf_matrix[idx_a],
                tfidf_matrix[idx_b]
            )[0, 0]

            coverage_jaccard, coverage_containment, overlap_country_count = (
                coverage_scores(
                    row_a["available_countries"],
                    row_b["available_countries"]
                )
            )

            shared = shared_terms(
                row_a["indicator_clean"] + " " + row_a["definition_clean"],
                row_b["indicator_clean"] + " " + row_b["definition_clean"]
            )

            shared_count = len([
                term
                for term in shared.split(";")
                if term.strip() != ""
            ])

            unit_match = (
                "Yes"
                if row_a["unit_clean"].lower() == row_b["unit_clean"].lower()
                and row_a["unit_clean"] != ""
                else "No"
            )

            # Broad cross-source flagging rule.
            # A pair is kept if it has meaningful textual similarity,
            # meaningful country coverage overlap, or strong containment.
            keep_pair = (
                (
                    indicator_similarity >= min_indicator_similarity
                    or text_similarity >= min_text_similarity
                )
                and shared_count >= min_shared_terms
                and (
                    coverage_jaccard >= min_coverage_jaccard
                    or coverage_containment >= min_coverage_containment
                    or overlap_country_count >= min_overlap_country_count
                )
            )

            if not keep_pair:
                continue

            pair_id += 1

            cross_source_rows.append({
                "pair_id": pair_id,
                "source_pair": dataset_a + "-" + dataset_b,

                "dataset_a": row_a["dataset"],
                "original_a": row_a["original"],
                "no_a": row_a["no"],
                "indicator_a": row_a["indicator"],
                "definition_a": row_a["Definition"],
                "unit_a": row_a["unit"],
                "source_a": row_a["source"],
                "country_count_a": row_a["country_count"],

                "dataset_b": row_b["dataset"],
                "original_b": row_b["original"],
                "no_b": row_b["no"],
                "indicator_b": row_b["indicator"],
                "definition_b": row_b["Definition"],
                "unit_b": row_b["unit"],
                "source_b": row_b["source"],
                "country_count_b": row_b["country_count"],

                "indicator_similarity_no_timing": round(indicator_similarity, 3),
                "text_similarity_no_timing": round(float(text_similarity), 3),
                "coverage_jaccard": round(coverage_jaccard, 3),
                "coverage_containment": round(coverage_containment, 3),
                "overlap_country_count": overlap_country_count,
                "unit_match": unit_match,
                "shared_terms": shared,
                "timing_terms_a": row_a["timing_terms"],
                "timing_terms_b": row_b["timing_terms"],
                "review_note": (
                    "Cross-source candidate. Use text similarity plus "
                    "coverage overlap and unit match to decide."
                )
            })

cross_source_df = pd.DataFrame(cross_source_rows)

print("Cross-source candidate shape:")
print(cross_source_df.shape)

Datasets found:
['imf', 'unsd', 'wb']
Comparing: imf vs unsd
Rows: 1089 x 158


imf vs unsd:   0%|          | 0/1089 [00:00<?, ?it/s]

Comparing: imf vs wb
Rows: 1089 x 76


imf vs wb:   0%|          | 0/1089 [00:00<?, ?it/s]

Comparing: unsd vs wb
Rows: 158 x 76


unsd vs wb:   0%|          | 0/158 [00:00<?, ?it/s]

Cross-source candidate shape:
(728, 28)


In [51]:
# ============================================================
# 10. Sort and save cross-source candidates
# ============================================================

if not cross_source_df.empty:
    cross_source_df = (
        cross_source_df
        .sort_values(
            by=[
                "coverage_containment",
                "coverage_jaccard",
                "text_similarity_no_timing",
                "indicator_similarity_no_timing",
                "overlap_country_count"
            ],
            ascending=[
                False,
                False,
                False,
                False,
                False
            ]
        )
        .reset_index(drop=True)
    )

    cross_source_df["pair_id"] = range(1, len(cross_source_df) + 1)

cross_source_df.to_csv(
    output_file,
    index=False
)

print("Saved:")
print(output_file)

print("Final shape:")
print(cross_source_df.shape)

print("Preview:")
print(cross_source_df.head())

Saved:
cross_source_overlap_candidates_recreated.csv
Final shape:
(728, 28)
Preview:
   pair_id source_pair dataset_a original_a  no_a  \
0        1    imf-unsd       imf   imf_0306   540   
1        2    imf-unsd       imf   imf_0309   543   
2        3    imf-unsd       imf   imf_0223   457   
3        4    imf-unsd       imf   imf_0223   457   
4        5    imf-unsd       imf   imf_0205   439   

                                         indicator_a  \
0  Climate related disasters frequency, Number of...   
1  Climate related disasters frequency, Number of...   
2  CO2 emissions in gross fixed capital formation...   
3  CO2 emissions in gross fixed capital formation...   
4  CO2 emissions in gross fixed capital formation...   

                                        definition_a     unit_a  \
0                                                NaN  Number of   
1                                                NaN  Number of   
2  Environment, Climate Change, Cross-Border, Dir...      

#### b.  Human Review and Exclusion Decision

After generating the automatic similarity assessment, we used the results as a **screening tool for human review**, not as an automatic deletion rule. The automatic assessment provided similarity scores, country-coverage overlap, unit comparison, shared terms, and explanatory notes. These outputs helped identify which indicator pairs required closer inspection.

The human review used the file `cross_source_overlap_candidates_evaluated` as the main review document. In this file, each potential overlap pair was assessed using the automatic explanation as supporting evidence. We then checked whether the indicators were truly redundant in meaning, unit, definition, and country coverage. The review was intentionally conservative: indicators were excluded only when they showed almost exact similarity and when retaining both would likely create duplication in the final dataset.

The review focused especially on cases where the apparent difference was limited to measurement timing, wording, or source-specific naming, rather than a substantive difference in the concept being measured.  

**Final Exclusion Decision**

Based on this conservative human review, only the following indicators were selected for exclusion:
un_0038
un_0037
un_0038
un_0037
un_0040

In [57]:
# ============================================================
# Exclude selected indicators from combined country availability
# ============================================================

import pandas as pd

# Input and output files
input_file = "combined_country_availability_wide.csv"
output_file = "combined_country_availability_final.csv"

# Indicators to exclude
exclude_ids = [
    "un_0038",
    "un_0037",
    "un_0038",
    "un_0037",
    "un_0040"
]

# Keep unique exclusion IDs
exclude_ids = sorted(set(exclude_ids))

# Load data
df = pd.read_csv(input_file, low_memory=False)

print("Original row count:")
print(len(df))

# Check which column contains the indicator IDs
possible_id_columns = ["original", "no", "indicator_id", "id"]

id_column = None

for col in possible_id_columns:
    if col in df.columns:
        matched_count = df[col].astype(str).isin(exclude_ids).sum()

        if matched_count > 0:
            id_column = col
            break

if id_column is None:
    raise ValueError(
        "Could not find the exclusion IDs in columns: "
        + ", ".join(possible_id_columns)
        + "\nAvailable columns are:\n"
        + ", ".join(df.columns)
    )

print("Using ID column:")
print(id_column)

print("Rows to exclude:")
print(df[id_column].astype(str).isin(exclude_ids).sum())

# Exclude rows
df_clean = df.loc[
    ~df[id_column].astype(str).isin(exclude_ids)
].copy()

print("Final row count:")
print(len(df_clean))

print("Rows removed:")
print(len(df) - len(df_clean))

# Save output
df_clean.to_csv(output_file, index=False)

print("Saved cleaned file:")
print(output_file)

Original row count:
1323
Using ID column:
original
Rows to exclude:
3
Final row count:
1320
Rows removed:
3
Saved cleaned file:
combined_country_availability_final.csv
